# Video Text Spotting from scratch — Colab / Kaggle

Trains a video text spotter with **no pretrained weights**. Runs on a free
Colab T4 or Kaggle P100.

**The constraint that shapes everything here:** Colab and Kaggle sessions are
capped at ~12 h (Kaggle also caps GPU at 30 h/week), and a full stage-1 run is
longer than that. So every config below writes a checkpoint every few hundred
steps to persistent storage, and you re-run the same cell with `--resume` after
a disconnect. Losing a session costs minutes, not the run.

> **Kaggle:** turn on *Settings → Accelerator → GPU* and *Settings → Internet → On*
> (internet is off by default and `pip install` will fail without it).


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Get the code and install

torch/torchvision are preinstalled on both platforms — installing only the rest
avoids a multi-GB reinstall that can also break the CUDA build.


In [ ]:
!git clone \
    https://github.com/Rahul5914/Ocr_thesis.git vtspot_repo
%cd vtspot_repo
!pip install -q opencv-python-headless shapely pyclipper einops

# More fonts = better recognition generalisation. matplotlib's 40 bundled
# faces are used automatically as a fallback, so this is optional.
!apt-get -qq install -y fonts-dejavu fonts-liberation fonts-freefont-ttf > /dev/null 2>&1 || true

from vtspot.data.synth_static import discover_fonts
print('fonts available:', len(discover_fonts()))


## 3. Verify everything works (~1 min)

Run this before starting a long job. It catches a broken environment in a
minute instead of at hour three.


In [ ]:
!python -m pytest tests/ -q
!python tools/train.py --config configs/smoke.yaml --device cpu --ckpt-dir /tmp/smoke


## 4. Persistent storage

**Colab:** mount Drive — `/content` is wiped when the session ends.
**Kaggle:** skip this cell; write to `/kaggle/working` (persists across sessions
in the same notebook, 20 GB limit).

Checkpoints are ~195 MB each. `keep_last_n: 3` in the configs prunes old ones,
so a stage costs ~800 MB rather than 3.9 GB.


In [ ]:
import os
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    CKPT = '/kaggle/working/vtspot'
else:
    from google.colab import drive; drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/vtspot'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)


## 5. Stage 1 — synthetic stills

This is what replaces ImageNet: with no pretrained backbone, stroke and glyph
features have to come from somewhere, and this is it. It is the longest stage —
budget several sessions.

**Nothing to download.** Images are generated procedurally at ~17/s per core.


In [ ]:
!python tools/train.py --config configs/colab_stage1.yaml \
    --ckpt-dir $CKPT/stage1


### Resuming after a disconnect

Re-run this cell as many times as needed — it continues from the last epoch.


In [ ]:
!python tools/train.py --config configs/colab_stage1.yaml \
    --ckpt-dir $CKPT/stage1 --resume $CKPT/stage1/last.pt


### Watch the losses

`loss_det` should fall steadily. `loss_binary` sitting at 1.0 early is normal —
the DB binary map stays inert until the probability map crosses the threshold.
`grad_norm` of 50–80 on the first steps is expected from random init and should
drop below ~5 within a few hundred steps.


In [ ]:
import json
h = json.load(open(f'{CKPT}/stage1/history.json'))
for e in h[-10:]:
    print(f"ep{e['epoch']:3d}  det={e['loss_det']:.3f} "
          f"rec={e['loss_rec']:.3f} total={e['loss_total']:.3f} "
          f"grad={e['grad_norm']:.1f}")


## 6. Stage 2 — synthetic video (tracking switches on)

Initialises from stage 1. The contrastive tracking loss becomes active here;
`clip_len: 4` matters, because below 3 there are no meaningful long-range
positives and the association head learns nothing.


In [ ]:
!python tools/train.py --config configs/colab_stage2.yaml \
    --init $CKPT/stage1/last.pt --ckpt-dir $CKPT/stage2


## 7. Stage 3 — real annotated video (optional)

Needs a downloaded benchmark. ICDAR2015-video needs a free account at
[rrc.cvc.uab.es](https://rrc.cvc.uab.es/?ch=3); upload it to Drive or attach it
as a Kaggle dataset, then convert:

**Read the validation output.** It compares your parsed density against the
published figure. If it says `SUSPICIOUS`, the parser matched the wrong format
variant and every number you produce afterwards is meaningless.


In [ ]:
!python tools/prepare_dataset.py --dataset icdar15_video \
    --videos $CKPT/raw/icdar15/videos \
    --annotations $CKPT/raw/icdar15/gt \
    --out $CKPT/data/icdar15_video_train


In [ ]:
!python tools/train.py --config configs/stage3_finetune.yaml \
    --init $CKPT/stage2/last.pt --ckpt-dir $CKPT/stage3 \
    --batch-size 2 --workers 2


## 8. Run it on a video

Outputs one entry per trajectory: track id, the aggregated transcription, the
frame span, and the polygon in every frame.


In [ ]:
!python tools/predict_video.py --checkpoint $CKPT/stage2/last.pt \
    --video sample.mp4 --out results.json --render annotated.mp4

import json
for t in sorted(json.load(open('results.json')), key=lambda d: -len(d['frames']))[:10]:
    print(f"id={t['track_id']:<4} {t['text']!r:<18} conf={t['text_confidence']:.2f} "
          f"frames={len(t['frames'])}")


### Preview a frame


In [ ]:
import cv2, json
from vtspot.predictor import draw_trajectories
from matplotlib import pyplot as plt
cap = cv2.VideoCapture('sample.mp4'); cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
ok, frame = cap.read(); cap.release()
if ok:
    vis = draw_trajectories(frame, json.load(open('results.json')), 10)
    plt.figure(figsize=(14, 8)); plt.axis('off')
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.show()


## 9. Evaluate

Prints tracking-mode (IoU only) **and** spotting-mode (IoU + correct
transcription) metrics. Quote the spotting numbers against video-text-*spotting*
literature — the gap between the two is large.


In [ ]:
!python tools/evaluate.py --checkpoint $CKPT/stage3/last.pt \
    --data $CKPT/data/icdar15_video_test --out $CKPT/results.json


---
## Memory reference

Peak RSS measured on CPU in fp32 — treat as an upper bound; a GPU with
`amp: true` uses roughly half.

| Config | Params | Peak (CPU fp32) |
|---|---|---|
| `stage1_static` (w48, 640, 8×1) | 12.2M | 8.8 GB |
| `stage2_video` (w48, 640, 2×4) | 12.2M | 8.8 GB |
| `colab_stage1/2` (w32, 512) | 8.6M | 5.9 GB |
| w32, 448, 2×3 | 8.6M | 3.8 GB |

A 16 GB T4 or P100 handles the full-size configs with AMP. The `colab_*` configs
trade a little capacity for headroom and faster epochs.

If you hit OOM: lower `crop_size` first, then `backbone_width`, then
`batch_size`. Keep `clip_len >= 3` — that one is load-bearing for tracking.
